In [1]:
import datetime
import numpy as np
import pandas as pd

import pickle

from utils import tf_idf

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
REPORTS_DATA_FILE = "<your path for isw dataset>"

In [4]:
#ISW_OUTPUT_DATA_FILE = "<>"
WEATHER_EVENTS_OUTPUT_DATA_FILE = "<your path for weather dataset>"
tfidf_transformer_model = "tfidf_transformer_"
count_vectorizer_model = "count_vectorizer_"

tfidf_transformer_version = "v1"
count_vectorizer_version = "v1"

In [5]:
def isNaN(num):
    return num != num

In [6]:
df_isw = pd.read_csv(REPORTS_DATA_FILE)

In [7]:
df_isw.head()

,Unnamed: 0.1,Unnamed: 0,id,date,short_url,title,text_title,full_url,html_content,html_content_v6,report_text_lemm,report_text_stemm
0,0,1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
1,1,1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
2,2,181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
3,3,197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
4,4,1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty e...,russian offen campaign assess februari twenti...


Prepare ISW for Merge

In [8]:
df_isw.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'id', 'date', 'short_url', 'title',
       'text_title', 'full_url', 'html_content', 'html_content_v6',
       'report_text_lemm', 'report_text_stemm'],
      dtype='object')

In [9]:
tfidf = pickle.load(open("tfidf_transformer.pkl", "rb"))
cv = pickle.load(open("count_vectorizer_v1.pkl", "rb"))

In [10]:
import numpy as np

feature_names = cv.get_feature_names_out()

def vector_to_keywords(tfidf_vector, feature_names, top_n=10):
    vector = tfidf_vector.toarray()[0]
    keywords_scores = {
        feature_names[i]: score
        for i, score in enumerate(vector) if score > 0
    }
    sorted_keywords = dict(sorted(keywords_scores.items(), key=lambda x: x[1], reverse=True)[:top_n])
    return sorted_keywords

In [11]:
corpus = df_isw['report_text_lemm'].fillna("").tolist()
X_counts = cv.transform(corpus)
X_tfidf = tfidf.transform(X_counts)

df_isw['keywords_vector'] = [
    vector_to_keywords(row, feature_names)
    for row in X_tfidf
]

In [12]:
df_isw['keywords_vector'].median

<bound method NDFrame._add_numeric_operations.<locals>.median of 0       {'februari': 0.46111364585068776, 'twenty': 0....
1       {'februari': 0.4046017827595065, 'twenty': 0.3...
2       {'februari': 0.4393921318104611, 'twenty': 0.3...
3       {'februari': 0.46287074916824467, 'twenty': 0....
4       {'februari': 0.4839400704975043, 'twenty': 0.3...
5       {'kyiv': 0.35674988512391226, 'envelop': 0.251...
6       {'kyiv': 0.46916007021450795, 'envelop': 0.280...
7       {'kyiv': 0.39762398397265836, 'chernihiv': 0.1...
8       {'kyiv': 0.3388343289039712, 'congest': 0.2137...
9       {'kyiv': 0.3302482081051005, 'hour': 0.2551019...
10      {'kyiv': 0.36879085148122026, 'lubni': 0.30245...
11      {'zaporizhya': 0.3179277436294645, 'kyiv': 0.2...
12      {'zaporizhya': 0.34810198212648547, 'kyiv': 0....
13      {'kyiv': 0.3115942802788235, 'mykolayiv': 0.17...
14      {'kyiv': 0.38148676174206136, 'ten': 0.3786245...
15      {'eleven': 0.48992752914271925, 'belarusian': ...
16     

In [13]:
df_isw.tail()

,Unnamed: 0.1,Unnamed: 0,id,date,short_url,title,text_title,full_url,html_content,html_content_v6,report_text_lemm,report_text_stemm,keywords_vector
1459,1459,736,737,2026-02-25,2026_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...,"{'februari': 0.4046017827595065, 'twenty': 0.3..."
1460,1460,827,828,2026-02-26,2026_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...,"{'februari': 0.4393921318104611, 'twenty': 0.3..."
1461,1461,1012,1013,2026-02-27,2026_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...,"{'februari': 0.46287074916824467, 'twenty': 0...."
1462,1462,470,471,2026-02-28,2026_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty e...,russian offen campaign assess februari twenti...,"{'februari': 0.4839400704975043, 'twenty': 0.3..."
1463,1463,1188,1189,2026-03-01,2026_03_01,"Russian Offensive Campaign Assessment, March 1...","Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 1...",russian offens campaign ass march isw option ...,russian offen campaign assess march isw optio...,"{'kyiv': 0.35674988512391226, 'envelop': 0.251..."


In [14]:
df_isw["date_datetime"] = pd.to_datetime(df_isw["date"])

In [15]:
df_isw['date_tomorrow_datetime'] = df_isw['date_datetime'].apply(lambda x: x+datetime.timedelta(days=1))

In [16]:
df_isw = df_isw.rename(columns = {"date_datetime":"report_date"})
#df_isw.to_csv(f"{ISW_OUTPUT_DATA_FILE}", sep=";", index=False)

Prepare Events for Merge

In [17]:
EVENTS_DATA_FILE = "<your path for alarms dataset>"

In [18]:
df_events = pd.read_csv(EVENTS_DATA_FILE, sep=";")

In [19]:
df_events.columns

Index(['id', 'merged_id', 'region_id', 'region_title', 'region_city',
       'all_region', 'start', 'end', 'original_alarms'],
      dtype='object')

In [20]:
df_events["type"] = "alarm"

In [21]:
df_events.columns

Index(['id', 'merged_id', 'region_id', 'region_title', 'region_city',
       'all_region', 'start', 'end', 'original_alarms', 'type'],
      dtype='object')

In [22]:
df_events_v2 = df_events.drop(["id","region_id","merged_id"],axis=1)

In [23]:
df_events_v2.head(5)

,region_title,region_city,all_region,start,end,original_alarms,type
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm
2,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,"[""52080""]",alarm
3,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,"[""52857""]",alarm
4,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,"[""52700""]",alarm


In [24]:
df_events_v2["start_time"] = df_events_v2.apply(lambda x: x["start"] if not isNaN(x["start"]) else x["event_time"] , axis=1)
df_events_v2["end_time"] = df_events_v2.apply(lambda x: x["end"] if not isNaN(x["end"]) else x["event_time"], axis=1)

In [25]:
df_events_v2["start_time"] = pd.to_datetime(df_events_v2["start"])
df_events_v2["end_time"] = pd.to_datetime(df_events_v2["end"])

In [26]:
df_events_v2.head()

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm,2022-02-24 14:00:43,2022-02-24 17:11:43
2,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,"[""52080""]",alarm,2022-02-24 15:40:42,2022-02-24 16:10:42
3,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,"[""52857""]",alarm,2022-02-24 20:11:47,2022-02-24 20:59:47
4,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,"[""52700""]",alarm,2022-02-25 01:59:36,2022-02-25 09:00:19


In [27]:
df_events_v2["start_hour"] = df_events_v2['start_time'].dt.floor('H')
df_events_v2["end_hour"] = df_events_v2['end_time'].dt.ceil('H')

In [28]:
df_events_v2.columns

Index(['region_title', 'region_city', 'all_region', 'start', 'end',
       'original_alarms', 'type', 'start_time', 'end_time', 'start_hour',
       'end_hour'],
      dtype='object')

In [29]:
df_events_v2["start_hour"] = df_events_v2["start_hour"].apply(lambda x: x if pd.notna(x) else None)
df_events_v2["end_hour"] = df_events_v2["end_hour"].apply(lambda x: x if pd.notna(x) else None)

In [30]:
df_events_v2["day_date"] = df_events_v2["start_time"].dt.date

df_events_v2["start_hour_datetimeEpoch"] = df_events_v2["start_hour"].apply(
    lambda x: int(x.timestamp()) if pd.notna(x) else None
)

df_events_v2["end_hour_datetimeEpoch"] = df_events_v2["end_hour"].apply(
    lambda x: int(x.timestamp()) if pd.notna(x) else None
)
df_events_v2.head()

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time,start_hour,end_hour,day_date,start_hour_datetimeEpoch,end_hour_datetimeEpoch
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645686000,1645696800
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm,2022-02-24 14:00:43,2022-02-24 17:11:43,2022-02-24 14:00:00,2022-02-24 18:00:00,2022-02-24,1645711200,1645725600
2,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,"[""52080""]",alarm,2022-02-24 15:40:42,2022-02-24 16:10:42,2022-02-24 15:00:00,2022-02-24 17:00:00,2022-02-24,1645714800,1645722000
3,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,"[""52857""]",alarm,2022-02-24 20:11:47,2022-02-24 20:59:47,2022-02-24 20:00:00,2022-02-24 21:00:00,2022-02-24,1645732800,1645736400
4,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,"[""52700""]",alarm,2022-02-25 01:59:36,2022-02-25 09:00:19,2022-02-25 01:00:00,2022-02-25 10:00:00,2022-02-25,1645750800,1645783200


In [31]:
df_events_v2["day_date"] = df_events_v2["start_time"].dt.date

df_events_v2["start_hour_datetimeEpoch"] = df_events_v2['start_hour'].apply(lambda x: int(x.strftime('%s'))  if not isNaN(x) else None)
df_events_v2["end_hour_datetimeEpoch"] = df_events_v2['end_hour'].apply(lambda x: int(x.strftime('%s'))  if not isNaN(x) else None)

df_events_v2.head(10)

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time,start_hour,end_hour,day_date,start_hour_datetimeEpoch,end_hour_datetimeEpoch
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645678800,1645689600
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm,2022-02-24 14:00:43,2022-02-24 17:11:43,2022-02-24 14:00:00,2022-02-24 18:00:00,2022-02-24,1645704000,1645718400
2,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,"[""52080""]",alarm,2022-02-24 15:40:42,2022-02-24 16:10:42,2022-02-24 15:00:00,2022-02-24 17:00:00,2022-02-24,1645707600,1645714800
3,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,"[""52857""]",alarm,2022-02-24 20:11:47,2022-02-24 20:59:47,2022-02-24 20:00:00,2022-02-24 21:00:00,2022-02-24,1645725600,1645729200
4,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,"[""52700""]",alarm,2022-02-25 01:59:36,2022-02-25 09:00:19,2022-02-25 01:00:00,2022-02-25 10:00:00,2022-02-25,1645743600,1645776000
5,Вінницька область,Вінницька обл.,1,2022-02-25 04:01:42,2022-02-25 08:35:42,"[""52081""]",alarm,2022-02-25 04:01:42,2022-02-25 08:35:42,2022-02-25 04:00:00,2022-02-25 09:00:00,2022-02-25,1645754400,1645772400
6,Харківська область,Харківська обл.,1,2022-02-25 04:56:47,2022-02-25 05:40:47,"[""52858""]",alarm,2022-02-25 04:56:47,2022-02-25 05:40:47,2022-02-25 04:00:00,2022-02-25 06:00:00,2022-02-25,1645754400,1645761600
7,Чернігівська область,Чернігівська обл.,1,2022-02-25 06:46:43,2022-02-25 06:52:43,"[""53293""]",alarm,2022-02-25 06:46:43,2022-02-25 06:52:43,2022-02-25 06:00:00,2022-02-25 07:00:00,2022-02-25,1645761600,1645765200
8,Львівська область,Львівська обл.,1,2022-02-25 06:53:17,2022-02-25 07:56:28,"[""52431""]",alarm,2022-02-25 06:53:17,2022-02-25 07:56:28,2022-02-25 06:00:00,2022-02-25 08:00:00,2022-02-25,1645761600,1645768800
9,Київська область,Київ,0,2022-02-25 07:19:04,2022-02-25 07:49:04,"[""72852""]",alarm,2022-02-25 07:19:04,2022-02-25 07:49:04,2022-02-25 07:00:00,2022-02-25 08:00:00,2022-02-25,1645765200,1645768800


In [32]:
df_events_v2["type"].head(5)

0    alarm
1    alarm
2    alarm
3    alarm
4    alarm
Name: type, dtype: object

In [33]:
df_events_v2["type"].shape

(76232,)

In [34]:
df_events_v2[~(df_events_v2["type"]=="alarm")].shape

(0, 14)

In [35]:
df_events_v2[~(df_events_v2["type"]!="alarm")].head(5)

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time,start_hour,end_hour,day_date,start_hour_datetimeEpoch,end_hour_datetimeEpoch
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645678800,1645689600
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm,2022-02-24 14:00:43,2022-02-24 17:11:43,2022-02-24 14:00:00,2022-02-24 18:00:00,2022-02-24,1645704000,1645718400
2,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,"[""52080""]",alarm,2022-02-24 15:40:42,2022-02-24 16:10:42,2022-02-24 15:00:00,2022-02-24 17:00:00,2022-02-24,1645707600,1645714800
3,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,"[""52857""]",alarm,2022-02-24 20:11:47,2022-02-24 20:59:47,2022-02-24 20:00:00,2022-02-24 21:00:00,2022-02-24,1645725600,1645729200
4,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,"[""52700""]",alarm,2022-02-25 01:59:36,2022-02-25 09:00:19,2022-02-25 01:00:00,2022-02-25 10:00:00,2022-02-25,1645743600,1645776000


In [36]:
df_events_v2[df_events_v2["type"]=="alarm"].shape

(76232, 14)

In [37]:
#df_events_v2.to_csv("<>", index=False)

Prepare Weather for Merge

In [38]:
WEATHER_DATA_FILE = "<path for weather dataset>"

In [39]:
df_weather = pd.read_csv(WEATHER_DATA_FILE)
df_weather["day_datetime"] = pd.to_datetime(df_weather["day_datetime"])

In [40]:
df_weather.shape

(808944, 65)

In [41]:
weather_exclude = [
"day_feelslikemax",
"day_feelslikemin",
"day_sunriseEpoch",
"day_sunsetEpoch",
"day_description",
"city_latitude",
"city_longitude",
"city_address",
"city_timezone",
"city_tzoffset",
"day_feelslike",
"day_precipprob",
"day_snow",
"day_snowdepth",
"day_windgust",
"day_windspeed",
"day_winddir",
"day_pressure",
"day_cloudcover",
"day_visibility",
"day_conditions",
"day_icon",
"day_source",
"day_preciptype",
"day_stations",
"hour_icon",
"hour_source",
"hour_stations",
"hour_feelslike"
]

In [42]:
df_weather_v2 = df_weather.drop(weather_exclude, axis=1)

In [43]:
df_weather_v2.head(2)

,city_resolvedAddress,day_datetime,day_datetimeEpoch,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipcover,day_solarradiation,day_solarenergy,day_uvindex,day_sunrise,day_sunset,day_moonphase,hour_datetime,hour_datetimeEpoch,hour_temp,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_preciptype,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions
0,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,00:00:00,1645653600,2.4,89.18,0.8,0.0,0.0,0.1,0.2,['snow'],31.3,15.5,275.6,1020.0,0.0,91.5,0.0,NaN,0.0,Overcast
1,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,01:00:00,1645657200,2.4,87.90,0.6,0.0,0.0,0.0,0.2,['snow'],27.7,14.8,280.3,1021.0,0.2,88.2,0.0,NaN,0.0,Partially cloudy


In [44]:
df_weather_v2["city"] = df_weather_v2["city_resolvedAddress"].apply(lambda x: x.split(",")[0])
#df_weather_v2["city"] = df_weather_v2["city"].replace('Хмельницька область', "Хмельницький")

Merging data

In [45]:
df_regions = pd.read_csv("<your path for regions list>")

In [46]:
df_regions.head(5)

,region,center_city_ua,center_city_en,region_alt,region_id
0,АР Крим,Сімферополь,Simferopol,Крим,1
1,Вінницька,Вінниця,Vinnytsia,Вінниччина,2
2,Волинська,Луцьк,Lutsk,Волинь,3
3,Дніпропетровська,Дніпро,Dnipro,Дніпропетровщина,4
4,Донецька,Донецьк,Donetsk,Донеччина,5


In [47]:
df_weather_reg = pd.merge(df_weather_v2, df_regions, left_on="city",right_on="center_city_ua")

In [48]:
df_weather_reg.head(2)

,city_resolvedAddress,day_datetime,day_datetimeEpoch,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipcover,day_solarradiation,day_solarenergy,day_uvindex,day_sunrise,day_sunset,day_moonphase,hour_datetime,hour_datetimeEpoch,hour_temp,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_preciptype,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions,city,region,center_city_ua,center_city_en,region_alt,region_id
0,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,00:00:00,1645653600,2.4,89.18,0.8,0.0,0.0,0.1,0.2,['snow'],31.3,15.5,275.6,1020.0,0.0,91.5,0.0,NaN,0.0,Overcast,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3
1,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,01:00:00,1645657200,2.4,87.90,0.6,0.0,0.0,0.0,0.2,['snow'],27.7,14.8,280.3,1021.0,0.2,88.2,0.0,NaN,0.0,Partially cloudy,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3


In [49]:
df_weather_reg.shape

(773376, 42)

In [50]:
df_weather_v2.shape

(808944, 37)

Merging weather and events

In [51]:
df_events_v2.dtypes

region_title                        object
region_city                         object
all_region                           int64
start                               object
end                                 object
original_alarms                     object
type                                object
start_time                  datetime64[ns]
end_time                    datetime64[ns]
start_hour                  datetime64[ns]
end_hour                    datetime64[ns]
day_date                            object
start_hour_datetimeEpoch             int64
end_hour_datetimeEpoch               int64
dtype: object

In [52]:
df_events_v2.shape

(76232, 14)

In [53]:
df_events_v2.head(2)

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time,start_hour,end_hour,day_date,start_hour_datetimeEpoch,end_hour_datetimeEpoch
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645678800,1645689600
1,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,"[""53292""]",alarm,2022-02-24 14:00:43,2022-02-24 17:11:43,2022-02-24 14:00:00,2022-02-24 18:00:00,2022-02-24,1645704000,1645718400


In [54]:
events_dict = df_events_v2.to_dict('records')
events_by_hour = []

In [55]:
events_dict[0]

{'region_title': 'Львівська область',
 'region_city': 'Львівська обл.',
 'all_region': 1,
 'start': '2022-02-24 07:43:17',
 'end': '2022-02-24 09:52:28',
 'original_alarms': '["52432"]',
 'type': 'alarm',
 'start_time': Timestamp('2022-02-24 07:43:17'),
 'end_time': Timestamp('2022-02-24 09:52:28'),
 'start_hour': Timestamp('2022-02-24 07:00:00'),
 'end_hour': Timestamp('2022-02-24 10:00:00'),
 'day_date': datetime.date(2022, 2, 24),
 'start_hour_datetimeEpoch': 1645678800,
 'end_hour_datetimeEpoch': 1645689600}

In [56]:
for event in events_dict:
    for d in pd.date_range(start=event["start_hour"], end=event["end_hour"], freq='1H'):
        et = event.copy()
        et["hour_level_event_time"] = d
        events_by_hour.append(et)

In [57]:
df_events_v3 = pd.DataFrame.from_dict(events_by_hour)

In [58]:
df_events_v3["hour_level_event_datetimeEpoch"] = df_events_v3["hour_level_event_time"].apply(
    lambda x: int(x.timestamp()) if pd.notna(x) else None
)

In [59]:
df_events_v3.head(2)

,region_title,region_city,all_region,start,end,original_alarms,type,start_time,end_time,start_hour,end_hour,day_date,start_hour_datetimeEpoch,end_hour_datetimeEpoch,hour_level_event_time,hour_level_event_datetimeEpoch
0,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645678800,1645689600,2022-02-24 07:00:00,1645686000
1,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,"[""52432""]",alarm,2022-02-24 07:43:17,2022-02-24 09:52:28,2022-02-24 07:00:00,2022-02-24 10:00:00,2022-02-24,1645678800,1645689600,2022-02-24 08:00:00,1645689600


In [60]:
df_weather_reg.head(5)

,city_resolvedAddress,day_datetime,day_datetimeEpoch,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipcover,day_solarradiation,day_solarenergy,day_uvindex,day_sunrise,day_sunset,day_moonphase,hour_datetime,hour_datetimeEpoch,hour_temp,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_preciptype,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions,city,region,center_city_ua,center_city_en,region_alt,region_id
0,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,00:00:00,1645653600,2.4,89.18,0.8,0.0,0.0,0.1,0.2,['snow'],31.3,15.5,275.6,1020.0,0.0,91.5,0.0,NaN,0.0,Overcast,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3
1,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,01:00:00,1645657200,2.4,87.90,0.6,0.0,0.0,0.0,0.2,['snow'],27.7,14.8,280.3,1021.0,0.2,88.2,0.0,NaN,0.0,Partially cloudy,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3
2,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,02:00:00,1645660800,2.9,88.58,1.2,0.0,0.0,0.0,0.1,['snow'],29.2,14.4,310.0,1022.0,10.0,100.0,NaN,NaN,NaN,Overcast,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3
3,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,03:00:00,1645664400,2.3,86.63,0.3,0.0,0.0,0.0,0.1,['snow'],23.8,13.3,295.1,1021.0,0.1,92.0,0.0,NaN,0.0,Overcast,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3
4,"Луцьк, Луцький район, Україна",2022-02-24,1645653600,4.9,0.7,2.6,0.0,83.7,0.118,4.17,36.9,2.8,1.0,07:13:36,17:51:06,0.77,04:00:00,1645668000,1.9,87.85,0.1,0.0,0.0,0.0,0.1,['snow'],24.5,13.3,305.8,1021.0,0.0,93.8,0.0,NaN,0.0,Overcast,Луцьк,Волинська,Луцьк,Lutsk,Волинь,3


In [61]:
df_events_v4 = df_events_v3.copy().add_prefix('event_')

In [62]:
df_weather_v4 = df_weather_reg.merge(df_events_v4, 
                                     how="left", 
                                     left_on=["region_alt","hour_datetimeEpoch"],
                                     right_on=["event_region_city","event_hour_level_event_datetimeEpoch"])

In [63]:
#df_weather_v4.to_csv("<>", sep=";", index=False)

In [64]:
df_weather_v4.shape

(773376, 58)